<a href="https://colab.research.google.com/github/Willian-devprog/Colecao_de.Filmes/blob/main/Colecao_de.Filmes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 Coleção de Filmes e Séries — Sistema de Gerenciamento e Avaliação

Este projeto consiste em uma aplicação em **Python** desenvolvida para execução no **Google Colab/VS Code**, criada como projeto prático para a disciplina de **Programação I** da UFRPE. O sistema permite que usuários gerenciem suas coleções de filmes e séries, registrem avaliações com comentários e analisem seus hábitos de consumo de mídias sem a necessidade de um banco de dados externo, utilizando persistência em arquivos **CSV**.

## 👥 Integrantes do Grupo e Responsabilidades
| Nome Completo | Responsabilidade no Projeto |
| :--- | :--- |
| **José Bezerra Cabral Neto** | Desenvolvimento das rotinas de autenticação (Login/Cadastro) e controle de permissões. |
| **Marcos A. Lira Gomes** | Implementação das funções de cadastro de mídias, importação CSV e filtro por palavra-chave. |
| **Willian Hassan Costa Santos** | Criação do sistema de avaliação (0-10 com comentários), ranking, recomendações e exportação de relatórios. |

## 🚀 Funcionalidades do Sistema
- **Autenticação e Controle de Acesso:**
  - **Consulta Aberta**: Qualquer pessoa pode buscar e visualizar mídias e avaliações sem precisar fazer login.
  - **Área do Usuário**: Para cadastrar novas mídias ou registrar avaliações, é obrigatório efetuar login no sistema.
- **Cadastro de Mídias (Filmes e Séries):**
  - Registro individual informando título, categoria/gênero, tipo (Filme ou Série) e ano de lançamento.
- **Importação de Coleção Pessoal (CSV):**
  - Permite importar de forma automatizada um arquivo `.csv` contendo uma lista prévia de filmes e séries.
- **Avaliações com Comentários:**
  - Atribuição de notas em um intervalo fechado de **0.0 a 10.0** (onde `0` representa *Não recomendaria* e `10` representa *Super recomendaria / Perfeito*).
  - Espaço dedicado para inclusão de comentários e opiniões pessoais sobre a mídia.
- **Busca por Palavra-Chave:**
  - Pesquisa rápida de mídias por título, gênero ou tipo.
- **Ranking Geral:**
  - Listagem das mídias mais bem avaliadas pelos usuários organizadas por nota.
- **Recomendações Personalizadas:**
  - Algoritmo simples de recomendação baseado nos gêneros mais consumidos e bem avaliados pelo usuário logado.
- **Exportação de Relatórios:**
  - Geração de relatórios sumarizados em arquivo de texto (`.txt`) com estatísticas da coleção e histórico de avaliações.

## 🛠️ Tecnologias e Bibliotecas Utilizadas
- **Linguagem:** Python 3
- **Ambiente de Desenvolvimento:** Visual Studio Code (VS Code)
- **Ambiente de Execução Alternativo:** Google Colab
- **Persistência de Dados:** Arquivos `.csv` e `.txt` (sem uso de banco de dados relacional/SQL)
- **Bibliotecas Nativas do Python:**
  - `csv`: Para leitura e escrita estruturada dos dados da coleção e usuários.
  - `os` / `sys`: Para manipulação e verificação de arquivos e caminhos no ambiente Colab.
  
## 📋 Instruções de Execução Passo a Passo (Google Colab)
1. **Acessar o Notebook:**
   - Abra o arquivo `.ipynb` do projeto diretamente no Google Colab.
2. **Preparar os Arquivos de Dados:**
   - Certifique-se de carregar os arquivos `.csv` necessários na aba **Arquivos** (ícone de pasta no menu esquerdo do Colab) ou execute a célula inicial do código para que o programa crie automaticamente a estrutura de dados inicial.
3. **Executar a Aplicação:**
   - Execute a célula principal do projeto para iniciar o menu interativo via terminal/console do Colab.
4. **Navegação:**
   - Utilize as opções numéricas do menu para navegar entre as rotinas de consulta, login, cadastro, avaliação e exportação de relatórios.

## 📋 Instruções de Execução Passo a Passo (VS Code)
1. **Clonar o Repositório:**
``bash
    git clone https://github.com/Willian-devprog/Colecao_de.Filmes.git
    cd Colecao_de.Filmes

## 📂 Estrutura dos Arquivos de Dados (CSV)
O projeto armazena as informações em arquivos texto com separador `;` para garantir a persistência dos dados entre execuções:
   - `usuarios.csv`: Armazena os dados dos usuários cadastrados (`id;nome;login;senha`).
   - `midias.csv`: Contém o acervo de filmes e séries (`id;titulo;tipo;genero;ano`).
   - `avaliacoes.csv`: Registra o histórico de avaliações (`id_midia;id_usuario;nota;comentario`)
  


In [2]:

import csv
import json
import sqlite3
from datetime import datetime
from pathlib import Path

ARQUIVO = Path("filmes.json")
BANCO = Path("banco.db")

conexao = sqlite3.connect(BANCO)
cursor = conexao.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS contas (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL UNIQUE,
    senha TEXT NOT NULL
)
""")
conexao.commit()


def cadastrar_conta():
    nome = input("Nome: ").strip()
    senha = input("Senha: ").strip()

    if not nome or not senha:
        print("Nome e senha não podem ficar vazios.")
        return

    try:
        cursor.execute(
            "INSERT INTO contas (nome, senha) VALUES (?, ?)",
            (nome, senha)
        )
        conexao.commit()
        print("Conta criada!")

    except sqlite3.IntegrityError:
        print("Esse nome já existe.")


def login_conta():
    nome = input("Nome: ").strip()
    senha = input("Senha: ").strip()

    cursor.execute(
        "SELECT nome FROM contas WHERE nome = ? AND senha = ?",
        (nome, senha)
    )

    conta = cursor.fetchone()

    if conta:
        print(f"Bem-vindo, {conta[0]}!")
        return conta[0]

    print("Dados incorretos.")
    return None


def listar_contas():
    cursor.execute("SELECT id, nome FROM contas")
    contas = cursor.fetchall()

    if not contas:
        print("Nenhuma conta cadastrada.")
        return

    for id_conta, nome in contas:
        print(f"{id_conta} - {nome}")


def deletar_conta(usuario_atual):
    nome = input("Nome da conta para deletar: ").strip()

    cursor.execute(
        "DELETE FROM contas WHERE nome = ?",
        (nome,)
    )

    conexao.commit()

    if cursor.rowcount:
        print("Deletado.")

        if usuario_atual == nome:
            return None

    else:
        print("Conta não encontrada.")

    return usuario_atual


def carregar():
    # Lê os filmes salvos. Se não houver arquivo
    # ou ele estiver quebrado, começa com uma lista vazia.

    if not ARQUIVO.exists():
        return []

    try:
        with open(ARQUIVO, "r", encoding="utf-8") as f:
            filmes = json.load(f)

            if not isinstance(filmes, list):
                print("Formato inválido. Iniciando com lista vazia.")
                return []

            return filmes

    except json.JSONDecodeError:
        print("Arquivo corrompido. Iniciando com lista vazia.")
        return []

    except OSError as erro:
        print(f"Erro ao acessar o arquivo: {erro}")
        return []


def salvar(filmes):
    # Grava a lista inteira no arquivo JSON.

    try:
        with open(ARQUIVO, "w", encoding="utf-8") as f:
            json.dump(filmes, f, ensure_ascii=False, indent=2)

    except OSError as erro:
        print(f"Erro ao salvar os filmes: {erro}")


def validar_ano(texto):
    # Converte o ano para inteiro e confere se é um valor razoável.

    try:
        ano = int(texto)

    except ValueError:
        raise ValueError("O ano deve ser um número inteiro.")

    ano_maximo = datetime.now().year + 5

    # 1888 = primeiro filme da história
    # Permite filmes ainda não lançados.

    if not 1888 <= ano <= ano_maximo:
        raise ValueError(
            f"O ano deve estar entre 1888 e {ano_maximo}."
        )

    return ano


def validar_nota(texto):
    # Converte a nota para decimal entre 0.0 e 10.0.
    # Aceita '7,5' e '7.5'.

    try:
        nota = float(texto.replace(",", "."))

    except ValueError:
        raise ValueError(
            "A nota deve ser um número (ex: 8 ou 7.5)."
        )

    if not 0.0 <= nota <= 10.0:
        raise ValueError("A nota deve estar entre 0.0 e 10.0.")

    return round(nota, 1)


def criar_filme(titulo, categoria, ano, nota, comentario=""):
    # Valida todos os campos e devolve o dicionário do filme.

    titulo = titulo.strip()
    categoria = categoria.strip()
    ano = str(ano).strip()
    nota = str(nota).strip()

    if not titulo or not categoria or not ano or not nota:
        raise ValueError(
            "Título, categoria, ano e nota são obrigatórios."
        )

    return {
        "titulo": titulo,
        "categoria": categoria,
        "ano": validar_ano(ano),
        "nota": validar_nota(nota),
        "comentario": comentario.strip(),
    }


def filme_existe(filmes, titulo):
    # Verifica se já existe um filme com esse título
    # (ignora maiúsculas/minúsculas).

    return any(
        f["titulo"].lower() == titulo.strip().lower()
        for f in filmes
    )


def exibir(lista):
    # Mostra uma lista de filmes na tela
    # (usada na listagem e na busca).

    for i, f in enumerate(lista, 1):
        # .get() evita erro com filmes antigos que não tinham comentário.

        comentario = f.get("comentario", "")

        print(
            f"{i}. [{f['categoria']}] {f['titulo']} "
            f"({f['ano']}) - Nota: {f['nota']:.1f}"
        )

        if comentario:
            print(f"   Comentário: {comentario}")


def adicionar(filmes):
    # Opção 1: cadastra um filme, com nota e comentário.

    print("\n--- Adicionar novo filme ---")

    titulo = input("Título do filme: ").strip()

    if titulo and filme_existe(filmes, titulo):
        print(
            f"Atenção: o filme '{titulo}' "
            "já consta na sua coleção!"
        )
        return

    categoria = input(
        "Categoria (ex: Ação, Comédia, Terror): "
    )

    ano = input("Ano de lançamento: ")
    nota = input("Nota (0.0 a 10.0): ")
    comentario = input("Comentário (opcional): ")

    try:
        filme = criar_filme(
            titulo,
            categoria,
            ano,
            nota,
            comentario
        )

    except ValueError as erro:
        print(f"Erro: {erro}")
        return

    filmes.append(filme)
    salvar(filmes)

    print(
        f"Filme '{filme['titulo']}' adicionado com sucesso!"
    )


def listar(filmes):
    # Opção 2: mostra todos os filmes.

    if not filmes:
        print("\nSua coleção de filmes está vazia.")
        return

    print(
        f"\n--- Sua Coleção ({len(filmes)} filmes) ---"
    )

    exibir(filmes)


def ordenar(filmes):
    # Opção 3: ordena a coleção por título, ano ou nota.

    if not filmes:
        print("\nNenhum filme cadastrado para ordenar.")
        return

    print("\n--- Critérios de Ordenação ---")
    print("1. Por Título (A-Z)")
    print("2. Por Ano (mais antigo ao mais recente)")
    print("3. Por Nota (maior para menor)")

    criterio = input(
        "Escolha o critério (1-3): "
    ).strip()

    if criterio == "1":
        filmes.sort(
            key=lambda f: f["titulo"].lower()
        )

    elif criterio == "2":
        filmes.sort(
            key=lambda f: f["ano"]
        )

    elif criterio == "3":
        filmes.sort(
            key=lambda f: f["nota"],
            reverse=True
        )

    else:
        print("Opção de ordenação inválida.")
        return

    salvar(filmes)
    print("Filmes ordenados com sucesso!")


def buscar(filmes):
    # Opção 4: procura a palavra no título ou na categoria.

    if not filmes:
        print("\nNenhum filme cadastrado para buscar.")
        return

    palavra = input(
        "\nDigite a palavra-chave: "
    ).strip().lower()

    if not palavra:
        print("Digite pelo menos uma letra para buscar.")
        return

    encontrados = [
        f for f in filmes
        if palavra in f["titulo"].lower()
        or palavra in f["categoria"].lower()
    ]

    if not encontrados:
        print(
            f"Nenhum filme encontrado para '{palavra}'."
        )
        return

    print(
        f"\n--- {len(encontrados)} resultado(s) "
        f"para '{palavra}' ---"
    )

    exibir(encontrados)


def importar_csv(filmes):
    print("\n--- Importar CSV ---")

    print(
        "Colunas esperadas: titulo, categoria, ano, "
        "nota, comentario (opcional)"
    )

    caminho = Path(
        input("Caminho do arquivo .csv: ").strip().strip('"')
    )

    if not caminho.is_file():
        print("Erro: arquivo não encontrado.")
        return

    importados = 0
    duplicados = 0
    erros = []

    # Guarda mensagens como:
    # "Linha 4: A nota deve ser..."

    try:
        with open(
            caminho,
            "r",
            encoding="utf-8-sig",
            newline=""
        ) as f:

            # Descobre o separador olhando só o cabeçalho:
            # ";" (Excel BR) ou ","

            cabecalho = f.readline()

            separador = (
                ";"
                if cabecalho.count(";") > cabecalho.count(",")
                else ","
            )

            f.seek(0)

            # Volta ao início para o leitor processar o arquivo inteiro.

            leitor = csv.DictReader(
                f,
                delimiter=separador
            )

            # Normaliza os nomes das colunas
            # (minúsculas, sem espaços).

            if leitor.fieldnames:
                leitor.fieldnames = [
                    c.strip().lower()
                    for c in leitor.fieldnames
                ]

            obrigatorias = {
                "titulo",
                "categoria",
                "ano",
                "nota"
            }

            if (
                not leitor.fieldnames
                or not obrigatorias.issubset(leitor.fieldnames)
            ):
                print(
                    "Erro: o cabeçalho deve conter as colunas: "
                    "titulo, categoria, ano, nota."
                )
                return

            # A linha 1 é o cabeçalho,
            # então os dados começam na linha 2.

            for numero_linha, linha in enumerate(leitor, start=2):

                titulo = (
                    linha.get("titulo") or ""
                ).strip()

                if titulo and filme_existe(filmes, titulo):
                    duplicados += 1
                    continue

                try:
                    filme = criar_filme(
                        titulo,
                        linha.get("categoria") or "",
                        linha.get("ano") or "",
                        linha.get("nota") or "",
                        linha.get("comentario") or "",
                    )

                except ValueError as erro:
                    erros.append(
                        f"Linha {numero_linha}: {erro}"
                    )
                    continue

                filmes.append(filme)
                importados += 1

    except (OSError, csv.Error, UnicodeError) as erro:
        print(f"Erro ao ler o arquivo CSV: {erro}")
        return

    if importados:
        salvar(filmes)

    print("\n--- Resultado da importação ---")
    print(f"Importados: {importados}")
    print(f"Ignorados por já existirem: {duplicados}")
    print(f"Com erro: {len(erros)}")

    for mensagem in erros:
        print(f"  - {mensagem}")


# AQUI EMBAIXO ESTÁ O MENU PRINCIPAL

def gerenciar_filmes(usuario):
    filmes = carregar()

    while True:
        print("\n GERENCIADOR DE COLEÇÃO DE FILMES ")
        print("1. Adicionar novo filme")
        print("2. Ver lista de filmes")
        print("3. Ordenar filmes")
        print("4. Buscar por palavra-chave")
        print("5. Importar de CSV")
        print("6. Trocar de conta / Sair do gerenciador")

        opcao = input(
            "\nEscolha uma opção (1-6): "
        ).strip()

        if opcao == "1":
            adicionar(filmes)

        elif opcao == "2":
            listar(filmes)

        elif opcao == "3":
            ordenar(filmes)

        elif opcao == "4":
            buscar(filmes)

        elif opcao == "5":
            importar_csv(filmes)

        elif opcao == "6":
            print("Saindo da coleção...")
            break

        else:
            print("Opção inválida. Escolha de 1 a 6.")


def menu_acesso():
    usuario = None

    while True:
        print(
            f"\n[Logado: {usuario or 'ninguém'}]"
        )

        print(
            "1-Cadastrar \n2-Login \n3-Listar contas"
            "\n4-Deletar conta \n5-Acessar filmes \n0-Sair"
        )

        op = input("Opção: ").strip()

        if op == "1":
            cadastrar_conta()

        elif op == "2":
            usuario = login_conta()

        elif op == "3":
            listar_contas()

        elif op == "4":
            usuario = deletar_conta(usuario)

        elif op == "5":
            if usuario:
                gerenciar_filmes(usuario)
            else:
                print(
                    "Faça login primeiro para acessar os filmes."
                )

        elif op == "0":
            print("Encerrando programa...")
            break

        else:
            print("Opção inválida.")


if __name__ == "__main__":
    try:
        menu_acesso()

    finally:
        conexao.close()



[Logado: ninguém]
1-Cadastrar 
2-Login 
3-Listar contas
4-Deletar conta 
5-Acessar filmes 
0-Sair
Opção: 5
Faça login primeiro para acessar os filmes.

[Logado: ninguém]
1-Cadastrar 
2-Login 
3-Listar contas
4-Deletar conta 
5-Acessar filmes 
0-Sair
Opção: 1
Nome: Marcos
Senha: Lira
Conta criada!

[Logado: ninguém]
1-Cadastrar 
2-Login 
3-Listar contas
4-Deletar conta 
5-Acessar filmes 
0-Sair
Opção: 2
Nome: Marcos
Senha: Lira
Bem-vindo, Marcos!

[Logado: Marcos]
1-Cadastrar 
2-Login 
3-Listar contas
4-Deletar conta 
5-Acessar filmes 
0-Sair
Opção: 5

 GERENCIADOR DE COLEÇÃO DE FILMES 
1. Adicionar novo filme
2. Ver lista de filmes
3. Ordenar filmes
4. Buscar por palavra-chave
5. Importar de CSV
6. Trocar de conta / Sair do gerenciador

Escolha uma opção (1-6): 6
Saindo da coleção...

[Logado: Marcos]
1-Cadastrar 
2-Login 
3-Listar contas
4-Deletar conta 
5-Acessar filmes 
0-Sair
Opção: 6
Opção inválida.

[Logado: Marcos]
1-Cadastrar 
2-Login 
3-Listar contas
4-Deletar conta 
5-Acessa